In [1]:
import os
import json
from gigachat import GigaChat

In [2]:
token=os.getenv("GIGACHAT_CREDENTIALS","MDFhMDYyY2UtYzVlOS03NjFmLWI2YTQtZDg0MDhiOTg2YWNiOmY2ZDhmNjJkLTBjYzMtNDIxNC1iZmZhLTY1YzEwOTIyOWRkNA==")
model_name="GigaChat-3-Ultra"
db_file="db.json"

json_systen_prompt=(
    "Tы—строгий механизм извлечения данных для платформы «UrFU TeamUp».\n"
    "Проанализируйте текст, введенный пользователем, и определите тип его профиля.\n\n"
    "Важные правила:\n"
    "1.Отвечайте ТОЛЬКО «сырым» валидным JSON-объектом,точно соответствующим приведенной ниже схеме.\n"
    "2.НЕ пишите никакого текста в разговорном стиле до или после JSON.\n"
    "3.НЕ оборачивайте JSON в блоки кода Markdown, такие как ```json ... ```.\n\n"
    "Далее следует JSON-схема.:\n"
    "{\n"
    "  \"user_type\": \"student\" or \"project_leader\",\n"
    "  \"extracted_data\": {\n"
    "    \"name_or_title\": \"Name of the student OR Title of the project\",\n"
    "    \"core_skills_or_needs\": [\"list\", \"of\", \"skills\", \"or\", \"technologies\"],\n"
    "    \"summary_description\": \"A clean 1-sentence summary of who they are or what they build\"\n"
    "  }\n"
    "}"
)

# =====================================================================
# ФУНКЦИЯ 1:Извлекаем текст пользователя в реальном времени в структурированный словарь
# ===================================================================== 

def extract_profile_to_dict(user_input:str)->dict:
    """
    Отправляет в GigaChat необработанный ввод пользователя и принудительно запрашивает ответ в формате структурированного словаря Python.
    В случае сбоя разбора возвращает словарь с информацией об ошибке.
    """
    with GigaChat(credentials=token,verify_ssl_certs=False) as giga:
        payload=[
            {"role":"system","content":json_systen_prompt},
            {"role":"user","content":user_input}
        ]
        response=giga.chat({"messages":payload,"model":model_name})
        raw_output=response.choices[0].message.content.strip()
        
        if raw_output.startswith("```"):
            raw_output=raw_output.strip("`").replace("json","", 1).strip()
            
        try:
            return json.loads(raw_output)
        except json.JSONDecodeError:
            return {"error":"Invalid JSON received from AI","raw":raw_output}

# =====================================================================
# ФУНКЦИЯ 2:Автоматически добавляем структурированный профиль в локальное хранилище
# =====================================================================
def save_profile_to_db(profile_data:dict,filename:str=db_file):
    """
    Безопасно открывает файл db.json, добавляет данные созданного профиля. 
    и сохраняет массив списка обратно на диск с аккуратным форматированием.
    """
    if "error" in profile_data:
        return False

    if os.path.exists(filename):
        try:
            with open(filename,"r",encoding="utf-8") as file:
                database=json.load(file)
                if not isinstance(database,list):
                    database=[database]
        except (json.JSONDecodeError,ValueError):
            database=[]
    else:
        database=[]

    database.append(profile_data)

    with open(filename,"w",encoding="utf-8") as file:
        json.dump(database,file,indent=4,ensure_ascii=False)
    return True

# =====================================================================
# ФУНКЦИЯ 3:Двусторонний локальный мэтчинг (студенты<->тимлиды)
# =====================================================================
def query_matchmaker(search_context,str,look_for_type:str,filename:str=db_file)->str:
    """
    Выполняет запрос к локальному файлу базы данных,объединяет профили в строковый контекст,
    и дает GigaChat указание подбирать разработчиков для проектов или наоборот.
    
    :param search_context:Текущий поисковый запрос,введенный пользователем в поиске совпадений.
    :param look_for_type:Установливаем значение 'student' (если поиск ведет лид) или 'project_leader' (если поиск ведет студент)
    """
    if not os.path.exists(filename):
        return "База данных пуста.Зарегистрируйте первых участников!"

    with open(filename,"r",encoding="utf-8") as file:
        records=json.load(file)

    # Gather matching profile records from our local JSON list array
    target_pool=[r for r in records if r.get("user_type")==look_for_type]
    
    if not target_pool:
        type_label="разработчиков" if look_for_type=="student" else "открытых проектов"
        return f"В базе данных пока нет доступных {type_label}."

    pool_text=""
    for idx,item in enumerate(target_pool):
        data=item["extracted_data"]
        pool_text+=f"- Вариант #{idx+1}:{data['name_or_title']}.Стек/Требования:{','.join(data['core_skills_or_needs'])}.Описание:{data['summary_description']}\n"

    if look_for_type=="student":
        role_desc="Ты—ИИ-Рекрутер платформы UrFU TeamUp.Помоги Тимлиду найти разработчиков из базы данных."
        query_label="Запрос Тимлида (какие навыки нужны проекту)"
    else:
        role_desc="Ты—ИИ-Ментор платформы UrFU TeamUp.Помоги одиночному Студенту найти подходящую команду."
        query_label="Профиль и навыки Студента"

    hr_prompt=(
        f"{role_desc}\n"
        f"Наша текущая локальная база данных:\n{pool_text}\n\n"
        f"Входящий {query_label}:\"\"\"{search_context}\"\"\"\n\n"
        f"ЗАДАЧА:Выбери топ-1 или топ-2 лучших совпадения из предоставленной базы данных."
        f"Для каждого выбранного совпадения напиши короткое,убедительное предложение-обоснование (почему это идеальный мэтч)."
        f"Отвечай вежливо,профессионально,на русском языке."
    )

    with GigaChat(credentials=token,verify_ssl_certs=False) as giga:
        response=giga.chat({"messages":[{"role":"user","content":hr_prompt}],"model":model_name})
        return response.choices[0].message.content

In [3]:
import os
import json
import sys
from gigachat import GigaChat

# --- CONFIGURATION ---
# Replace with your working long credentials string from Sber Studio
TOKEN = os.getenv("GIGACHAT_CREDENTIALS", "MDFhMDYyY2UtYzVlOS03NjFmLWI2YTQtZDg0MDhiOTg2YWNiOmY2ZDhmNjJkLTBjYzMtNDIxNC1iZmZhLTY1YzEwOTIyOWRkNA==")
MODEL_NAME = "GigaChat-3-Ultra"  # or GigaChat-Max depending on your Studio tier
DB_FILE = "db.json"

# --- SYSTEM PROMPT (Strict JSON Extraction) ---
JSON_SYSTEM_PROMPT = (
    "You are a strict data-extraction engine for the 'UrFU TeamUp' platform.\n"
    "Analyze the user's input text and determine their profile type.\n\n"
    "CRITICAL RULES:\n"
    "1. Respond ONLY with a raw, valid JSON object matching the exact schema below.\n"
    "2. Do NOT write any conversational text before or after the JSON.\n"
    "3. Do NOT wrap the JSON inside markdown code blocks like ```json ... ```.\n\n"
    "JSON SCHEMA TO FOLLOW:\n"
    "{\n"
    "  \"user_type\": \"student\" or \"project_leader\",\n"
    "  \"extracted_data\": {\n"
    "    \"name_or_title\": \"Name of the student OR Title of the project\",\n"
    "    \"core_skills_or_needs\": [\"list\", \"of\", \"skills\", \"or\", \"technologies\"],\n"
    "    \"summary_description\": \"A clean 1-sentence summary of who they are or what they build\"\n"
    "  }\n"
    "}"
)

# ==========================================
# CORE FUNCTION 1: PROFILE EXTRACTION
# ==========================================
def extract_profile_to_dict(user_input: str) -> dict:
    """
    Sends raw user text to GigaChat and forces a structured Python dictionary output.
    """
    with GigaChat(credentials=TOKEN, verify_ssl_certs=False) as giga:
        payload = [
            {"role": "system", "content": JSON_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ]
        response = giga.chat({"messages": payload, "model": MODEL_NAME})
        raw_output = response.choices[0].message.content.strip()
        
        # Safe cleanup if markdown codeblock sneaks in
        if raw_output.startswith("```"):
            raw_output = raw_output.strip("`").replace("json", "", 1).strip()
            
        try:
            return json.loads(raw_output)
        except json.JSONDecodeError:
            return {"error": "Invalid JSON received from AI", "raw": raw_output}

# ==========================================
# CORE FUNCTION 2: AUTOMATED LOCAL STORAGE
# ==========================================
def save_profile_to_db(profile_data: dict, filename: str = DB_FILE):
    """
    Safely loads db.json array, appends the new profile structure, and writes it back.
    """
    if "error" in profile_data:
        print("⚠️ Database write aborted: Extraction error present.")
        return

    if os.path.exists(filename):
        try:
            with open(filename, "r", encoding="utf-8") as file:
                database = json.load(file)
                if not isinstance(database, list):
                    database = [database]
        except (json.JSONDecodeError, ValueError):
            database = []
    else:
        database = []

    database.append(profile_data)

    with open(filename, "w", encoding="utf-8") as file:
        json.dump(database, file, indent=4, ensure_ascii=False)
    print(f"💾 Saved to local database! (Current database size: {len(database)})")

# ==========================================
# CORE FUNCTION 3: TWO-WAY MATCHMAKING QUERY
# ==========================================
def query_matchmaker(search_context: str, look_for_type: str, filename: str = DB_FILE) -> str:
    """
    Reads db.json, builds a localized context, and asks GigaChat to pair up 
    Team Leads with Developers or vice versa.
    
    :param search_context: The query string (e.g., 'We need a Figma designer' or 'I am a backend coder')
    :param look_for_type: Either 'student' (if team lead is searching) or 'project_leader' (if student is searching)
    """
    if not os.path.exists(filename):
        return "База данных пуста. Зарегистрируйте первых участников!"

    with open(filename, "r", encoding="utf-8") as file:
        records = json.load(file)

    # Filter out only target entries from our clean database array
    target_pool = [r for r in records if r.get("user_type") == look_for_type]
    
    if not target_pool:
        type_label = "разработчиков" if look_for_type == "student" else "открытых проектов"
        return f"В базе данных пока нет доступных {type_label}."

    # Format the local DB records into plain text context for GigaChat
    pool_text = ""
    for idx, item in enumerate(target_pool):
        data = item["extracted_data"]
        pool_text += f"- Вариант #{idx+1}: {data['name_or_title']}. Стек/Требования: {', '.join(data['core_skills_or_needs'])}. Описание: {data['summary_description']}\n"

    # Dynamic system routing prompt based on who is searching
    if look_for_type == "student":
        role_desc = "Ты — ИИ-Рекрутер платформы UrFU TeamUp. Помоги Тимлиду найти разработчиков из базы данных."
        query_label = "Запрос Тимлида (какие навыки нужны проекту)"
    else:
        role_desc = "Ты — ИИ-Ментор платформы UrFU TeamUp. Помоги одиночному Студенту найти подходящую команду."
        query_label = "Профиль и навыки Студента"

    hr_prompt = (
        f"{role_desc}\n"
        f"Наша текущая локальная база данных:\n{pool_text}\n\n"
        f"Входящий {query_label}: \"\"\"{search_context}\"\"\"\n\n"
        f"ЗАДАЧА: Выбери топ-1 или топ-2 лучших совпадения из предоставленной базы данных. "
        f"Для каждого выбранного совпадения напиши короткое, убедительное предложение-обоснование (почему это идеальный мэтч). "
        f"Отвечай вежливо, профессионально, на русском языке."
    )

    with GigaChat(credentials=TOKEN, verify_ssl_certs=False) as giga:
        response = giga.chat({"messages": [{"role": "user", "content": hr_prompt}], "model": MODEL_NAME})
        return response.choices[0].message.content


# ==========================================
# LOCAL TESTING SIMULATION INTERFACE
# ==========================================
if __name__ == "__main__":
    print("====================================================")
    print("🚀 UrFU TeamUp Production Module Tests Initiated")
    print("====================================================")
    
    # --- PHASE 1: Populate Database with a Student ---
    print("\n[Симуляция 1]: Регистрация Студента-Дизайнера...")
    student_text = "Я Яра Сист, отлично владею Figma, собираю адаптивные веб-интерфейсы и готовлю презентации для защиты."
    parsed_student = extract_profile_to_dict(student_text)
    save_profile_to_db(parsed_student)

    print("\n[Симуляция 2]: Регистрация инженера машинного обучения...")
    student_text = "Я Дил Довочело,отлично владею python,имею опыт в разработке ии и работы с машинным обучением."
    parsed_student = extract_profile_to_dict(student_text)
    save_profile_to_db(parsed_student)
    
    # --- PHASE 2: Matchmaking Query for a Team Lead ---
    print("\n[Симуляция 3]: Тимлид ищет фронтенд/дизайнера...")
    teamlead_search = "Мы делаем интерактивную карту для кампуса Новокольцовский. Срочно ищем человека, который нарисует UI в Figma!"
    
    # We call query_matchmaker asking it to look into 'student' records
    recommendations = query_matchmaker(search_context=teamlead_search, look_for_type="student")
    
    print("\n--- РЕЗУЛЬТАТ ПОДБОРА ДЛЯ ТИМЛИДА ---")
    print(recommendations)
    print("====================================================")

🚀 UrFU TeamUp Production Module Tests Initiated

[Симуляция 1]: Регистрация Студента-Дизайнера...
💾 Saved to local database! (Current database size: 1)

[Симуляция 2]: Регистрация инженера машинного обучения...
💾 Saved to local database! (Current database size: 2)

[Симуляция 3]: Тимлид ищет фронтенд/дизайнера...

--- РЕЗУЛЬТАТ ПОДБОРА ДЛЯ ТИМЛИДА ---
Здравствуйте! Я проанализировал запрос вашего тимлида и базу данных. Для задачи по созданию UI интерактивной карты в Figma я подобрал следующих кандидатов:

**Топ-1 совпадение:**
*   **Кандидат:** Яра Сист.
*   **Обоснование:** Яра идеально подходит для этой роли, так как её профиль полностью сфокусирован на создании адаптивных интерфейсов в Figma — это именно тот навык, который срочно требуется вашему проекту.

**Топ-2 совпадение (резервный вариант):**
*   **Кандидат:** Дил Довочело.
*   **Обоснование:** Хотя основной стек Дила — Python и машинное обучение, его можно рассмотреть только в том случае, если помимо отрисовки UI потребуется с